The Loop is the stuctural failure: A standard python `for` loop processes one number at a time, forcing the CPU to wait on the Python Iterpreter for every single element. In real production manual loops are 100x to 1000x slower than equivalent numpy operations.

Numpy nypasses the interpreter by sending entire arrays to compiled C code using SIMD: Single Instruction Multiple Data hardware instructions, allowing the CPU to process multiple pair of numbers simultaneously.

Why I must Kill the LOOP?:

1. Array Thinking: Once you kill the loop, and adapt the Array Thinking mindset: every other domain - from PANDAS operations and SQL aggregations to PyTorch tensors and Transformers attention mechanism - feels like same architectural pattern.
2. Unlocking the GPU: The "Loop Killer" mindset is the only path to Deep Learning; GPU ML is only possible because operations are vectorized, not iterated
3.Ownership of Failure Modes: Looped thinking hides Broadcasting traps and Memory Explosions that only become visible when you begin to audit the shapes and memory footprints of your array
4.  A "Tutorial Consumer" writes instructions for a list; a "System Architect" defines spatial transformations (Ax+b) that act on entire data manifolds at once

# LOOP KILLER

In [ ]:
# Pairwise Distance: 10. Compute the Euclidian distance btw every pair of points in a 100 * 2 coordinate array.
import numpy as np

data_points = 100
array = np.random.rand(data_points, 2)
print(f"Shape of 'array': {array.shape}")

Shape of 'array': (100, 2)


### Using `%timeit` and `%%timeit`

`%timeit` is a [Jupyter/IPython magic command](https://ipython.readthedocs.io/en/stable/interactive/magics.html) that measures the execution time of a single line of Python code. `%%timeit` is used to measure the execution time of an entire cell. It automatically performs multiple runs and provides statistics like the best, worst, and average execution times.

Let's see an example.

In [ ]:
import numpy as np

# Example using %timeit for a single line
print('Timing a single line:')
%timeit np.arange(1000)**2


Timing a single line:
2.4 µs ± 73.6 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [ ]:
%%timeit
for i in range(1000):
  i = i **2

66.4 µs ± 1.15 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


As we can observe a Numpy Vectorised operation is much more faster than regular for loops. From this we can prove that the Numpy was built to make Python faster.

# 1. Sum of all elements of a list containing 1 million numbers.

In [ ]:
sum = 0
for i in range(1_000_000):
  sum += i

print(sum)

sum_vectorised = np.arange(1_000_000).sum()
print(sum_vectorised)

499999500000
499999500000


In [ ]:
# sum of all elements of a list containing 1 million numbers.
%%timeit
sum=0
for i in range(1_000_000):
  sum += i


69 ms ± 21.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%timeit np.arange(1_000_000).sum()

921 µs ± 31.9 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


What takes a loop miliseconds is done by Numpy vectorisation in micro-second

# 2.Scaling: Multiply every element in a large array by a scalar (e.g., * 2).



Task: Scaling an Array

Here, we'll multiply every element in a large array by a scalar (e.g., `* 2`). We will compare a traditional Python loop with a vectorized NumPy operation.

In [ ]:
import numpy as np
array = np.arange(100000)
scaled_list = []
for i in array:
  scaled_list.append(i*2)
print("Looped Scaling:", scaled_list)
array = np.arange(100000) # Re-initialize for each run
array * 2
print("Direct NUMPY Scaling",array)

In [ ]:
%%timeit
import numpy as np
array = np.arange(100000)
scaled_list = []
for i in array:
  scaled_list.append(i*2)

26.2 ms ± 5.32 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%%timeit
_array = np.arange(100000) # Re-initialize for each run
_array * 2

111 µs ± 567 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


We can clearly see that scaling is done significally faster with Numpy rather than normally iterating through 100000 values and then scaling them.

# 3. Filtering Identify all elements in an array the are greater than 0.5:


In [ ]:
import numpy as np
array = np.linspace(0,1,1000)
greater = []
for i in array:
  if i > 0.5:
    greater.append(i)
print("Looped Idetification", greater[0:6])
array = np.linspace(0,1,1000)
gt = array[array > 0.5]
print(gt[00:6])


Looped Idetification [np.float64(0.5005005005005005), np.float64(0.5015015015015015), np.float64(0.5025025025025025), np.float64(0.5035035035035035), np.float64(0.5045045045045045), np.float64(0.5055055055055055)]
[0.5005005  0.5015015  0.5025025  0.5035035  0.5045045  0.50550551]


In [ ]:
%%timeit
import numpy as np
array = np.linspace(0,1,1000)
greater = []
for i in array:
  if i > 0.5:
    greater.append(i)

130 µs ± 36 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [ ]:
%%timeit
array = np.linspace(0,1,1000)
array[array > 0.5]

15.6 µs ± 2.58 µs per loop (mean ± std. dev. of 7 runs, 100000 loops each)


Here we used a fancy indexing method where we pass a condition as an index so we will get the elements only where the condition is satisfied.

# 4. Row-wise aggetation: compute the mean of every row in a 1000 * 1000 matrix

In [ ]:
import numpy as np
matrix_size = 1000
matrix = np.random.rand(matrix_size, matrix_size)
row_means = []
for row in matrix:
  row_means.append((np.sum(row))/(len(row))) # Not callable error that's why used np.sum instead of sum

print("Looped Mean:", row_means[0:6])

matrix_size = 1000
matrix = np.random.rand(matrix_size, matrix_size)
row_mean = matrix.mean(axis=1)
print("Numpy Mean:", row_mean[0:6])

Looped Mean: [np.float64(0.49775650763761553), np.float64(0.49920333153214225), np.float64(0.4982647543131222), np.float64(0.49281411827471516), np.float64(0.4914292320047275), np.float64(0.4962960732002982)]
Numpy Mean: [0.48966358 0.51239214 0.49508834 0.49964528 0.49270548 0.49673766]


In [ ]:
%%timeit
import numpy as np
matrix_size = 1000
matrix = np.random.rand(matrix_size, matrix_size)
row_means = []
for row in matrix:
  row_means.append((np.sum(row))/(len(row)))

13.9 ms ± 2.1 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
%%timeit
import numpy as np
matrix_size = 1000
matrix = np.random.rand(matrix_size, matrix_size)
row_mean = matrix.mean(axis=1)

10.3 ms ± 2.04 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


Calculated means of 1000 rows and used axis = 1 while doing in Numpy.

# 5. Normalisation

In [ ]:
import numpy as np
matrix_size = 1000
matrix = np.random.rand(matrix_size, matrix_size) # Re-initialize for each run
normalised_matrix_list = []
for row in matrix:
  min_val = np.min(row)
  max_val = np.max(row)
  # Handle cases where max_val equals min_val to avoid division by zero
  if (max_val - min_val) == 0:
      normalised_row = np.zeros_like(row) # Or fill with 0.5, or 1.0, depending on desired behavior
  else:
      normalised_row = (row - min_val) / (max_val - min_val)
  normalised_matrix_list.append(normalised_row)
print("Normalised Matrix:", np.array(normalised_matrix_list)[0:5, 0:5]) ## GREAT THIS WAS A BIG ARRAY.

matrix_size = 1000
matrix = np.random.rand(matrix_size, matrix_size) # Re-initialize for each run
matrix = (matrix - matrix.min())/ ((matrix.max() - matrix.min()))
print("Numpy Normalisation", matrix[0:5])

Normalised Matrix: [[0.65821381 0.5480023  0.38778805 0.32885529 0.00324067]
 [0.45765829 0.11999771 0.61323276 0.64456498 0.68425087]
 [0.55995809 0.24302319 0.78178933 0.84138479 0.32005429]
 [0.25489343 0.77646134 0.73973521 0.48519443 0.02282778]
 [0.98724155 0.89233163 0.02196673 0.06548813 0.99586954]]
Numpy Normalisation [[0.32728414 0.42682319 0.83872343 ... 0.9909513  0.01958749 0.21485881]
 [0.17196518 0.27830981 0.50797993 ... 0.1660188  0.62464394 0.12806622]
 [0.2361058  0.2299496  0.99762396 ... 0.89760129 0.99720585 0.33450407]
 [0.50553105 0.90433721 0.82231848 ... 0.96735759 0.06946399 0.00181185]
 [0.97593402 0.77804954 0.44023337 ... 0.64111123 0.60451819 0.97548623]]


In [ ]:
%%timeit
import numpy as np
matrix_size = 1000
matrix = np.random.rand(matrix_size, matrix_size) # Re-initialize for each run
normalised_matrix_list = []
for row in matrix:
  min_val = np.min(row)
  max_val = np.max(row)
  # Handle cases where max_val equals min_val to avoid division by zero
  if (max_val - min_val) == 0:
      normalised_row = np.zeros_like(row) # Or fill with 0.5, or 1.0, depending on desired behavior
  else:
      normalised_row = (row - min_val) / (max_val - min_val)
  normalised_matrix_list.append(normalised_row)

21.6 ms ± 614 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [1]:
%%timeit
import numpy as np
matrix_size = 1000
matrix = np.random.rand(matrix_size, matrix_size) # Re-initialize for each run
matrix = (matrix - matrix.min())/ ((matrix.max() - matrix.min()))

15.9 ms ± 3.1 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


Normalization in machine learning is a preprocessing technique used to scale numerical features into a specific, uniform range (typically 0 to 1). By using normalisation with Numpy it saves you from iterating over such a large dataset and saves time making the process faster.

In [3]:
%%timeit
import numpy as np
matrix_size = 1000
_matrix_for_row_norm = np.random.rand(matrix_size, matrix_size) # Fresh matrix for each run
row_min = _matrix_for_row_norm.min(axis=1, keepdims=True)
row_max = _matrix_for_row_norm.max(axis=1, keepdims=True)
normalized_row_wise_matrix = (_matrix_for_row_norm - row_min) / (row_max - row_min)
# Handle cases where row_max equals row_min to avoid division by zero
normalized_row_wise_matrix = np.nan_to_num(normalized_row_wise_matrix, nan=0.0)

42.4 ms ± 18.5 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


# 6.  Manual STD vs np.std


In [4]:
import numpy as np
array = np.arange(1_00_000)
len = array.size
total_sum = 0
for value in array:
  total_sum += value
mean = total_sum / len

squared_diff_sum = 0
for value in array:
  diff = value - mean
  squared_diff = diff **2
  squared_diff_sum += squared_diff

variance = squared_diff_sum / len
std_deviation  = variance ** 0.5

print("Looped STD:", std_deviation)

array = np.arange(1_00_000)
array = array.std()
print("Numpy STD:", array)

Looped STD: 28867.513458037913
Numpy STD: 28867.513458037913


In [5]:
%%timeit
import numpy as np
array = np.arange(1_00_000)
len = array.size
total_sum = 0
for value in array:
  total_sum += value
mean = total_sum / len

squared_diff_sum = 0
for value in array:
  diff = value - mean
  squared_diff = diff **2
  squared_diff_sum += squared_diff


variance = squared_diff_sum / len

std_deviation  = variance ** 0.5

53.5 ms ± 1.5 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
%%timeit
array = np.arange(1_00_000)
array = array.std()

449 µs ± 11.9 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


Numpy provides you a built-in .std() method. It helps us save time by not manually calculating the Standard deviation

Here also you can see, A loop takes milisecond what Numpy speed does in microseconds

# 7. Element wise power: Square every number

In [12]:
import numpy as np
array = np.random.rand(1_000)
squared_array = []
for i in array:
  squared_array.append(i ** 2)
print("Looped Square:", squared_array[0:6])
array = array ** 2
print("Numpy Squared", array[0:6])

Looped Square: [np.float64(0.015582574591452578), np.float64(0.2717659097831854), np.float64(0.016923875928897228), np.float64(0.02330712820499129), np.float64(0.7019441533371732), np.float64(0.2882871898434006)]
Numpy Squared [0.01558257 0.27176591 0.01692388 0.02330713 0.70194415 0.28828719]


In [ ]:
%%timeit
import numpy as np
array = np.random.rand(1_000_000)
squared_array = []
for i in array:
  squared_array.append(i ** 2)

329 ms ± 10.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%%timeit
array = np.random.rand(1_000_000) ** 2

11.9 ms ± 2.57 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


This squaring was done on 1_000_000 numbers, and proved why we should use Numpy for mathematical and faster implementations.

# 8. Rolling Average: Calculate a moving window average across a time-series array

This task involves computing the average of a fixed-size window that slides over a sequence of data. We'll compare a Python loop approach with a more efficient NumPy vectorized method, typically using `np.convolve` or `np.lib.stride_tricks.sliding_window_view`.

In [ ]:
import numpy as np
array_size = 1_000_000
window_size = 10
_time_series_array = np.random.rand(array_size) # Re-initialize for each run
_rolling_averages_py = []
# Changed len(_time_series_array) to _time_series_array.shape[0] to avoid variable shadowing
for i in range(_time_series_array.shape[0] - window_size + 1):
    window = _time_series_array[i : i + window_size]
    _rolling_averages_py.append(np.mean(window)) # Using np.mean for the window as it's efficient
print("Python loop for rolling average:", _rolling_averages_py[0:6])

import numpy as np
# array_size = 1_000_000
# window_size = 10
# _time_series_array = np.random.rand(array_size) # Re-initialize for each run
_weights = np.ones(window_size) / window_size
_rolling_averages_np = np.convolve(_time_series_array, _weights, mode='valid')
print("Vectorized NumPy for rolling average (np.convolve):", _rolling_averages_np[0:6])


Python loop for rolling average: [np.float64(0.5396810153671472), np.float64(0.5724765515725523), np.float64(0.6099660059788949), np.float64(0.5960783255637861), np.float64(0.6327054306452989), np.float64(0.6105495892809856)]
Vectorized NumPy for rolling average (np.convolve): [0.53968102 0.57247655 0.60996601 0.59607833 0.63270543 0.61054959]


In [14]:
%%timeit
import numpy as np
array_size = 1_000_000
window_size = 10
_time_series_array = np.random.rand(array_size) # Re-initialize for each run
_rolling_averages_py = []
# Changed len(_time_series_array) to _time_series_array.shape[0] to avoid variable shadowing
for i in range(_time_series_array.shape[0] - window_size + 1):
    window = _time_series_array[i : i + window_size]
    _rolling_averages_py.append(np.mean(window)) # Using np.mean for the window as it's efficient
# print("Python loop for rolling average:")

6.99 s ± 698 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


 Vectorized NumPy for Rolling Average (using `np.convolve`)

NumPy's `np.convolve` can efficiently calculate rolling sums (and thus averages) by using an array of ones as weights. The `mode='valid'` ensures that only positions where the window fully overlaps are considered.

In [13]:
%%timeit
import numpy as np
array_size = 1_000_000
window_size = 10
_time_series_array = np.random.rand(array_size) # Re-initialize for each run
_weights = np.ones(window_size) / window_size
_rolling_averages_np = np.convolve(_time_series_array, _weights, mode='valid')
# print("Vectorized NumPy for rolling average (np.convolve):")

19.1 ms ± 3.93 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


Observed a significant speedup with the `np.convolve` vectorized approach compared to the Python loop for calculating the rolling average, especially for large arrays.

Conclusion: As Numpy stands for Numerical-Python, it is a really important and foundational library that provides you so many fundamental, faster and useful methods. By providing the concept known as Vectorisation: performing mathematical operations directly on the entire arrays rather than iterating through individual elements using explicit `for` loop